# QSR Ops: 00 · Synthetic operational tables

Generates the **operational tables** that the Today, Labor, and Inventory tabs read. Run this
notebook before the app. The Guest Sentiment tables come from `notebooks/sentiment/01` to `03`.

| Table | Grain | Read by |
|-------|-------|---------|
| `dims_stores` | 1 row per store (20) | Store names, cities, square footage |
| `dims_items` | 1 row per SKU (50) | Inventory categories and item names |
| `dims_employees` | 1 row per crew member (240) | Crew roster and pay rates |
| `facts_sales_daypart` | store, date, daypart (60 days) | Today KPIs, Labor forecast cards |
| `facts_labor_daypart` | store, date, daypart, role (60 days) | Labor plan and labor % |
| `facts_sales_inventory_daily` | store, date, SKU (60 days) | Inventory health, watched items |
| `facts_purchase_orders` | PO line | Staged reorders |
| `facts_customer_feedback` | 1 row per guest response (1,000) | Wiring banner row counts |

The data is **synthetic** and the brand is **fictional**. The generator is seeded, so the same
`seed` and `anchor_date` always produce the same rows. Each run overwrites the eight tables.

> **Env:** set the `catalog` and `schema` widgets to a catalog and schema you own. Point the app at
> the same place with `CC_CATALOG` and `CC_SCHEMA` in `app/app.yaml`. Keep `anchor_date` equal to
> `ANCHOR_DATE` in that file, because the Today, Labor, and Inventory queries are anchored on it.

In [ ]:
# --- Parameters (widgets) ---------------------------------------------------
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "qsr_ops_demo", "Schema")
dbutils.widgets.text("anchor_date", "2026-06-22", "Anchor date (match ANCHOR_DATE in app/app.yaml)")
dbutils.widgets.text("seed", "2026", "Random seed (same seed, same data)")

import datetime as dt
import math
import random
from decimal import Decimal

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
ANCHOR = dt.date.fromisoformat(dbutils.widgets.get("anchor_date").strip())
SEED = int(dbutils.widgets.get("seed").strip())
if not CATALOG or not SCHEMA:
    raise ValueError("Set the catalog and schema widgets.")
FQ = lambda t: f"{CATALOG}.{SCHEMA}.{t}"

try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
except Exception as e:  # some compute does not let you change the session time zone
    print(f"Session time zone not changed ({type(e).__name__}); timestamps are stored as naive UTC.")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Target: {CATALOG}.{SCHEMA}   anchor={ANCHOR}   seed={SEED}")

In [ ]:
# --- Reference data (fictional) ---------------------------------------------
rng = random.Random(SEED)

HISTORY_DAYS = 60  # sales, labor, inventory: anchor-59 .. anchor
FEEDBACK_DAYS = 61  # guest feedback: anchor-60 .. anchor
DATES = [ANCHOR - dt.timedelta(days=HISTORY_DAYS - 1 - i) for i in range(HISTORY_DAYS)]
FEEDBACK_DATES = [ANCHOR - dt.timedelta(days=FEEDBACK_DAYS - 1 - i) for i in range(FEEDBACK_DAYS)]

# Same city order as store_locations in notebooks/sentiment/01: S001 is the first store, and so on.
CITIES = [
    ("Detroit", "MI"), ("Detroit", "MI"), ("Warren", "MI"), ("Chicago", "IL"), ("Chicago", "IL"),
    ("Houston", "TX"), ("Houston", "TX"), ("Phoenix", "AZ"), ("Las Vegas", "NV"), ("Atlanta", "GA"),
    ("Tampa", "FL"), ("Indianapolis", "IN"), ("Columbus", "OH"), ("Charlotte", "NC"), ("Dallas", "TX"),
    ("Memphis", "TN"), ("Cleveland", "OH"), ("Cincinnati", "OH"), ("Pittsburgh", "PA"), ("San Antonio", "TX"),
]
REGION = {"MI": "Central", "IL": "Central", "TX": "Central", "IN": "Central", "OH": "Central",
          "TN": "Central", "AZ": "West", "NV": "West", "GA": "East", "FL": "East", "NC": "East", "PA": "East"}

# Dayparts: name, hour_start, hour_end, shift hours, share of daily sales, average ticket ($)
DAYPARTS = [
    ("breakfast", 6, 10, 4, 0.13, 22.0),
    ("lunch", 11, 14, 3, 0.52, 25.0),
    ("dinner", 17, 20, 3, 0.21, 27.0),
    ("late", 20, 23, 3, 0.14, 28.0),
]
ROLES = ["cook", "cashier", "lead", "manager"]
CREW_MIX = {"cook": 5, "cashier": 4, "lead": 2, "manager": 1}  # 12 people per store
PAY_RANGE = {"cook": (17.50, 20.45), "cashier": (16.32, 19.21), "lead": (20.02, 23.00), "manager": (27.15, 30.00)}
BASE_HEADCOUNT = {  # scheduled people per daypart at demand 1.0, before LABOR_MULT
    "breakfast": {"cook": 2, "cashier": 2, "lead": 1, "manager": 1},
    "lunch": {"cook": 5, "cashier": 4, "lead": 2, "manager": 1},
    "dinner": {"cook": 3, "cashier": 3, "lead": 2, "manager": 1},
    "late": {"cook": 2, "cashier": 2, "lead": 1, "manager": 1},
}
LABOR_MULT = 1.85
BASE_REVENUE_DAY = 15000.0  # per store at scale 1.0 (about 1,900 sq ft)
DOW_SALES = [0.88, 0.86, 0.90, 0.96, 1.12, 1.26, 1.06]  # Monday .. Sunday
DOW_USAGE = [0.90, 0.88, 0.92, 0.98, 1.10, 1.24, 1.05]

# Category: (min, max) retail price in $. Each category has one vendor.
CATEGORY_PRICE = {
    "beverage": (1.5, 23.0), "bread_cheese": (2.5, 30.0), "meat": (1.7, 27.0),
    "pantry": (3.6, 27.0), "veggie": (3.2, 33.0),
}
VENDORS = {  # category: (vendor name, lead time in days)
    "beverage": ("Northgate Beverage Co.", 2),
    "meat": ("Ridgeline Meat Supply", 2),
    "veggie": ("Harvest Fields Produce", 1),
    "bread_cheese": ("Oak Hollow Dough Co.", 3),
    "pantry": ("Summit Pantry Commissary", 3),
}
ITEM_NAMES = [  # (category, item name), 50 SKUs
    ("beverage", "Cola fountain syrup (5 gal)"), ("beverage", "Lemon-lime fountain syrup (5 gal)"),
    ("beverage", "Root beer fountain syrup (5 gal)"), ("beverage", "Lemonade concentrate (1 gal)"),
    ("beverage", "Iced tea concentrate (1 gal)"), ("beverage", "Sparkling water (case)"),
    ("beverage", "Bottled water (case)"), ("beverage", "Sports drink (case)"),
    ("beverage", "Cold brew concentrate (1 gal)"), ("beverage", "Whole milk (gal)"),
    ("bread_cheese", "Pizza dough balls (case)"), ("bread_cheese", "Breadstick dough (case)"),
    ("bread_cheese", "Deep dish dough (case)"), ("bread_cheese", "Hoagie rolls (case)"),
    ("bread_cheese", "Mozzarella, shredded (5 lb)"), ("bread_cheese", "Provolone slices (5 lb)"),
    ("bread_cheese", "Parmesan, grated (5 lb)"), ("bread_cheese", "Cheddar, shredded (5 lb)"),
    ("bread_cheese", "Ricotta (3 lb)"), ("bread_cheese", "Cheese bites mix (case)"),
    ("bread_cheese", "Garlic butter (1 gal)"), ("bread_cheese", "Sesame topping (1 lb)"),
    ("meat", "Pepperoni, sliced (10 lb)"), ("meat", "Italian sausage crumbles (10 lb)"),
    ("meat", "Ham, diced (5 lb)"), ("meat", "Bacon, cooked (5 lb)"),
    ("meat", "Chicken wings, bone-in (40 lb)"), ("meat", "Chicken strips, grilled (10 lb)"),
    ("meat", "Beef meatballs (10 lb)"), ("meat", "Smoked turkey (5 lb)"), ("meat", "Ground beef (10 lb)"),
    ("veggie", "Yellow onions (50 lb)"), ("veggie", "Green bell peppers (25 lb)"),
    ("veggie", "Roma tomatoes (25 lb)"), ("veggie", "Mushrooms, sliced (5 lb)"),
    ("veggie", "Romaine hearts (case)"), ("veggie", "Jalapenos, fresh (10 lb)"),
    ("veggie", "Red onions (25 lb)"), ("veggie", "Baby spinach (5 lb)"),
    ("veggie", "Hass avocado (case)"), ("veggie", "Lemons (case)"),
    ("pantry", "Pizza sauce (#10 can)"), ("pantry", "Marinara sauce (#10 can)"),
    ("pantry", "Black olives, sliced (#10 can)"), ("pantry", "Banana peppers (#10 can)"),
    ("pantry", "Pizza flour (50 lb)"), ("pantry", "Vegetable oil (35 lb)"),
    ("pantry", "Breadcrumbs (25 lb)"), ("pantry", "Dry oregano (1 lb)"), ("pantry", "Ranch dressing (1 gal)"),
]
assert len(ITEM_NAMES) == 50
PAR_CHOICES = [8, 10, 12, 14, 18, 20, 24, 30]
USAGE_TRENDS = ["+22% usage", "+18% usage", "+9% usage", "steady", "steady"]
CHANNELS = ["yelp", "google", "survey", "app", "social"]
CHANNEL_WEIGHTS = [30, 25, 20, 15, 10]
FIRST = ["Allison", "Noah", "Angie", "Daniel", "Cristian", "Devon", "Aisha", "Marco", "Priya", "Jordan",
         "Elena", "Tyler", "Maya", "Ethan", "Lena", "Omar", "Grace", "Luis", "Hannah", "Samuel",
         "Nora", "Isaac", "Chloe", "Mateo", "Ruby", "Caleb", "Sofia", "Ian", "Zara", "Felix"]
LAST = ["Hill", "Rhodes", "Henderson", "Wagner", "Santos", "Pratt", "Mensah", "Ruiz", "Patel", "Brooks",
        "Lindqvist", "Carter", "Nguyen", "Foster", "Okafor", "Haddad", "Reyes", "Morgan", "Kim", "Silva",
        "Bennett", "Dalton", "Ferreira", "Quinn", "Weaver", "Ahmed", "Brennan", "Cho", "Diaz", "Ellison"]
PRODUCTS = ["pizza", "breadsticks", "wings", "cheese bites", "deep dish", "stuffed crust"]
FEEDBACK_TEXT = {
    "pos": ["Great {product} at {city} today. The crew was on point.",
            "{product} at {city} was hot and fresh, exactly what I wanted.",
            "Fast pickup at {city} and the {product} was perfect.",
            "The manager at {city} checked on us. Solid service and a tasty {product}."],
    "neu": ["Quick visit at {city}. Got my {product} and left. Standard.",
            "The {product} at {city} was fine. Nothing special, nothing wrong."],
    "neg": ["Waited too long for my {product} at {city}.",
            "The {product} at {city} arrived cold and a side was missing.",
            "The drive-thru line barely moved at {city} and the {product} was dry.",
            "The dining room at {city} needed cleaning when we arrived."],
}
FEEDBACK_THEMES = {
    "pos": ["freshness", "friendly_staff", "speed", "accuracy"],
    "neu": ["other", "value"],
    "neg": ["speed", "accuracy", "cleanliness", "freshness"],
}
FEEDBACK_REPLY = {
    "pos": "Thanks for the kind words about our {city} location! We'll share this with the team.",
    "neu": "Thanks for the feedback about our {city} location. We're working on it.",
    "neg": "We're sorry about your visit to our {city} location. Our team is looking into it.",
}


def money(x):
    """Two-decimal Decimal for the decimal(38,18) money columns."""
    return Decimal(f"{x:.2f}")

In [ ]:
# --- Stores, dims, and crew ---------------------------------------------------
STORES = []
for n, (city, state) in enumerate(CITIES, start=1):
    sqft = rng.randint(1200, 2800)
    STORES.append({
        "store_id": f"S{n:03d}", "store_name": f"Store {n:04d} - {city}", "city": city,
        "state": state, "region": REGION[state], "square_footage": sqft, "scale": sqft / 1900.0,
    })
S1 = STORES[0]["store_id"]

ITEMS = []
for i, (cat, name) in enumerate(ITEM_NAMES, start=1):
    lo, hi = CATEGORY_PRICE[cat]
    retail = round(rng.uniform(lo, hi), 2)
    cost = round(retail * rng.uniform(0.35, 0.50), 2)
    ITEMS.append({"sku": f"SKU-{i:04d}", "category": cat, "item_name": name,
                  "retail": retail, "cost": cost, "par": rng.choice(PAR_CHOICES)})
ITEM = {it["sku"]: it for it in ITEMS}
CAT_SKUS = {cat: [it["sku"] for it in ITEMS if it["category"] == cat] for cat in VENDORS}

EMPLOYEES = []
pay = {}
emp_no = 0
for st in STORES:
    for role, count in CREW_MIX.items():
        for _ in range(count):
            emp_no += 1
            lo, hi = PAY_RANGE[role]
            rate = round(rng.uniform(lo, hi), 2)
            hire = dt.date(2019, 1, 1) + dt.timedelta(days=rng.randint(0, 2555))
            name = f"{rng.choice(FIRST)} {rng.choice(LAST)}"
            EMPLOYEES.append((f"E{emp_no:05d}", hire, money(rate), name, role, st["store_id"]))
            pay.setdefault((st["store_id"], role), []).append(rate)
RATE = {k: sum(v) / len(v) for k, v in pay.items()}

stores_rows = [(s["city"], s["region"], s["square_footage"], s["state"], s["store_id"], s["store_name"])
               for s in STORES]
items_rows = [(it["category"], money(it["cost"]), money(it["retail"]), it["sku"], it["item_name"])
              for it in ITEMS]
print(f"stores={len(stores_rows)} items={len(items_rows)} employees={len(EMPLOYEES)}")

In [ ]:
# --- Sales and labor by daypart ----------------------------------------------
sales_rows, labor_rows = [], []
for st in STORES:
    sid, scale = st["store_id"], st["scale"]
    for d in DATES:
        wd = d.weekday()
        demand = DOW_SALES[wd] * scale
        for dp_name, hs, he, shift, share, ticket in DAYPARTS:
            expected_rev = BASE_REVENUE_DAY * demand * share
            actual_rev = expected_rev * rng.uniform(0.93, 1.07)
            fcst_rev = expected_rev * rng.uniform(0.95, 1.05)
            tkt = ticket * rng.uniform(0.97, 1.03)
            traffic = round(actual_rev / tkt)
            fcst_traffic = round(fcst_rev / tkt)
            sales_rows.append((d, dp_name, money(fcst_rev), fcst_traffic, he, hs,
                               money(actual_rev), sid, traffic))

            for role in ROLES:
                rate = RATE[(sid, role)]
                if role == "manager":
                    f_hc = 2 if (dp_name in ("lunch", "dinner") and demand > 1.3) else 1
                else:
                    f_hc = max(1, round(BASE_HEADCOUNT[dp_name][role] * LABOR_MULT * demand
                                        * rng.uniform(0.95, 1.05)))
                act_hc = max(0, f_hc + rng.choice([-1, 0, 0, 0, 1]))
                if role == "manager" and rng.random() < 0.2:
                    act_hc = 0
                hours = act_hc * shift
                f_cost = f_hc * shift * rate
                a_cost = hours * rate * rng.uniform(0.98, 1.03)
                labor_rows.append((d, dp_name, f_hc, money(f_cost), act_hc, money(a_cost), role, sid, hours))

print(f"sales rows={len(sales_rows)} labor rows={len(labor_rows)}")

In [ ]:
# --- Daily stock simulation (inventory) ---------------------------------------
series = {}  # (store_id, sku) -> {date: [on_hand_eod, units_sold]}
for st in STORES:
    sid, scale = st["store_id"], st["scale"]
    for it in ITEMS:
        par = it["par"]
        base_use = par * rng.uniform(0.12, 0.28) * math.sqrt(scale)
        on = int(par * rng.uniform(1.5, 2.8))
        days = {}
        for d in DATES:
            units = max(1, round(base_use * DOW_USAGE[d.weekday()] * rng.uniform(0.7, 1.3)))
            on = max(0, on - units)
            if on < par and rng.random() < 0.8:  # delivery lands most days a SKU runs under par
                on = max(on, 2 * par + rng.randint(0, par))
            days[d] = [on, units]
        series[(sid, it["sku"])] = days

# Anchor day at S001: six SKUs (three veggie, three meat) under par, the rest at or above par.
# These drive the watched list and the two staged reorders.
FORCED = rng.sample(CAT_SKUS["veggie"], 3) + rng.sample(CAT_SKUS["meat"], 3)
for it in ITEMS:
    cell = series[(S1, it["sku"])][ANCHOR]
    if it["sku"] in FORCED:
        cell[0] = max(1, int(it["par"] * rng.uniform(0.15, 0.55)))
    else:
        cell[0] = max(cell[0], it["par"])

inv_rows = []
for st in STORES:
    sid = st["store_id"]
    for it in ITEMS:
        for d in DATES:
            on, units = series[(sid, it["sku"])][d]
            inv_rows.append((d, on, it["par"], money(units * it["retail"]), it["sku"], sid, units))
print(f"inventory rows={len(inv_rows)}")

In [ ]:
# --- Purchase orders -----------------------------------------------------------
PO_DRAFTS = []


def add_po(sid, cat, skus, created, status, forced=False):
    vendor, lead = VENDORS[cat]
    day = created.date()
    lines = []
    for sku in skus:
        it = ITEM[sku]
        on = series[(sid, sku)][day][0]
        qty = it["par"] * 2 - on if forced else rng.randint(4, 24)
        lines.append({"sku": sku, "qty": qty, "on_hand": on, "par": it["par"],
                      "unit_cost": it["cost"], "trend": rng.choice(USAGE_TRENDS)})
    PO_DRAFTS.append({"created": created, "store_id": sid, "cat": cat, "vendor": vendor,
                      "lead": lead, "status": status, "lines": lines})


# S001: exactly two open reorders on the anchor day, one veggie and one meat vendor.
add_po(S1, "veggie", [s for s in FORCED if ITEM[s]["category"] == "veggie"],
       dt.datetime.combine(ANCHOR, dt.time(15, 0)), "staged", forced=True)
add_po(S1, "meat", [s for s in FORCED if ITEM[s]["category"] == "meat"],
       dt.datetime.combine(ANCHOR, dt.time(16, 0)), "staged", forced=True)

# Other stores: a few delivered and released orders from the past three weeks, plus sometimes one staged order today.
for st in STORES[1:]:
    sid = st["store_id"]
    for _ in range(rng.randint(3, 6)):
        k = rng.randint(1, 20)
        cat = rng.choice(list(VENDORS))
        skus = rng.sample(CAT_SKUS[cat], rng.randint(1, 3))
        created = dt.datetime.combine(ANCHOR - dt.timedelta(days=k),
                                      dt.time(rng.randint(8, 17), rng.choice([0, 15, 30, 45])))
        add_po(sid, cat, skus, created, "released" if k <= 2 else "delivered")
    if rng.random() < 0.5:
        cat = rng.choice(list(VENDORS))
        skus = rng.sample(CAT_SKUS[cat], rng.randint(1, 3))
        add_po(sid, cat, skus, dt.datetime.combine(ANCHOR, dt.time(rng.randint(13, 17), 0)), "staged")

PO_DRAFTS.sort(key=lambda p: (p["created"], p["store_id"]))
po_rows = []
for n, po in enumerate(PO_DRAFTS, start=1):
    po_id = f"PO-{n:06d}"
    eta = dt.datetime.combine(po["created"].date() + dt.timedelta(days=po["lead"]),
                              dt.time(rng.randint(8, 18), 0))
    for ln in po["lines"]:
        po_rows.append((po["created"], eta, po["lead"], money(ln["qty"] * ln["unit_cost"]),
                        ln["on_hand"], ln["par"], po_id, ln["qty"], ln["sku"], po["status"],
                        po["store_id"], money(ln["unit_cost"]), ln["trend"], po["cat"], po["vendor"]))
print(f"purchase order lines={len(po_rows)} orders={len(PO_DRAFTS)}")

In [ ]:
# --- Guest feedback (operational copy; the sentiment pipeline has its own reviews) ---
feedback_rows = []
for i in range(1, 1001):
    st = rng.choices(STORES, weights=[s["scale"] for s in STORES])[0]
    d = rng.choice(FEEDBACK_DATES)
    channel = rng.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0]
    sent = rng.choices(["pos", "neu", "neg"], weights=[60, 20, 20])[0]
    rating = {"pos": rng.choice([4, 5]), "neu": 3, "neg": rng.choice([1, 2])}[sent]
    nps = None
    if channel == "survey":
        nps = {"pos": rng.randint(40, 100), "neu": rng.randint(-10, 30), "neg": rng.randint(-100, -20)}[sent]
    needs_reply = sent == "neg" and rng.random() < 0.7
    theme = rng.choice(FEEDBACK_THEMES[sent])
    text = rng.choice(FEEDBACK_TEXT[sent]).format(product=rng.choice(PRODUCTS), city=st["city"])
    reply = FEEDBACK_REPLY[sent].format(city=st["city"])
    feedback_rows.append((channel, d, f"FB-{i:06d}", needs_reply, nps, rating, sent,
                          st["store_id"], theme, text, reply))

In [ ]:
# --- Write the eight tables (overwrite, so re-runs are idempotent) -------------
DDL = {
    "dims_stores": "city string, region string, square_footage bigint, state string, store_id string, store_name string",
    "dims_items": "category string, cost decimal(38,18), retail_price decimal(38,18), sku string, item_name string",
    "dims_employees": "employee_id string, hire_date date, hourly_rate decimal(38,18), name string, role string, store_id string",
    "facts_customer_feedback": "channel string, date date, feedback_id string, needs_reply boolean, nps bigint, rating bigint, sentiment_label string, store_id string, theme string, feedback_text string, ai_drafted_reply string",
    "facts_sales_daypart": "date date, daypart string, forecast_revenue decimal(38,18), forecast_traffic bigint, hour_end bigint, hour_start bigint, revenue decimal(38,18), store_id string, traffic bigint",
    "facts_labor_daypart": "date date, daypart string, forecast_headcount bigint, forecast_labor_cost decimal(38,18), headcount bigint, labor_cost decimal(38,18), role string, store_id string, total_hours bigint",
    "facts_sales_inventory_daily": "date date, on_hand_eod bigint, reorder_point bigint, revenue decimal(38,18), sku string, store_id string, units_sold bigint",
    "facts_purchase_orders": "created_at timestamp, eta timestamp, lead_time_days bigint, line_total decimal(38,18), on_hand_at_creation bigint, par bigint, po_id string, qty bigint, sku string, status string, store_id string, unit_cost decimal(38,18), usage_trend string, vendor_category string, vendor_name string",
}
TABLES = {
    "dims_stores": stores_rows,
    "dims_items": items_rows,
    "dims_employees": EMPLOYEES,
    "facts_customer_feedback": feedback_rows,
    "facts_sales_daypart": sales_rows,
    "facts_labor_daypart": labor_rows,
    "facts_sales_inventory_daily": inv_rows,
    "facts_purchase_orders": po_rows,
}
for name, rows in TABLES.items():
    df = spark.createDataFrame(rows, schema=DDL[name])
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(FQ(name)))
    print(f"{FQ(name)}: {len(rows):,} rows")

In [ ]:
# --- Self-check: the numbers the app queries depend on -------------------------
def scalar(sql):
    return spark.sql(sql).collect()[0][0]


anchor = str(ANCHOR)
checks = [
    ("dims_stores rows", scalar(f"SELECT COUNT(*) FROM {FQ('dims_stores')}"), 20),
    ("dims_items rows", scalar(f"SELECT COUNT(*) FROM {FQ('dims_items')}"), 50),
    ("facts_sales_daypart rows, anchor, S001", scalar(
        f"SELECT COUNT(*) FROM {FQ('facts_sales_daypart')} WHERE store_id = '{S1}' AND date = DATE'{anchor}'"), 4),
    ("inventory latest date", scalar(f"SELECT CAST(MAX(date) AS STRING) FROM {FQ('facts_sales_inventory_daily')}"), anchor),
    ("SKUs below par, S001, anchor", scalar(
        f"SELECT COUNT(*) FROM {FQ('facts_sales_inventory_daily')} WHERE store_id = '{S1}' "
        f"AND date = DATE'{anchor}' AND on_hand_eod < reorder_point"), 6),
    ("staged POs, S001", scalar(
        f"SELECT COUNT(DISTINCT po_id) FROM {FQ('facts_purchase_orders')} WHERE store_id = '{S1}' AND status = 'staged'"), 2),
]
for label, got, want in checks:
    status = "ok" if str(got) == str(want) else "MISMATCH"
    print(f"{status:9s} {label}: {got} (expected {want})")
assert all(str(got) == str(want) for _, got, want in checks), "self-check failed"

yday = scalar(f"SELECT SUM(revenue) FROM {FQ('facts_sales_daypart')} WHERE store_id = '{S1}' AND date = DATE'{anchor}' - 1")
labor_pct = scalar(
    f"SELECT ROUND(SUM(l.labor_cost) / SUM(s.revenue) * 100, 1) FROM "
    f"(SELECT SUM(revenue) AS revenue FROM {FQ('facts_sales_daypart')} WHERE store_id = '{S1}' AND date = DATE'{anchor}') s "
    f"CROSS JOIN (SELECT SUM(labor_cost) AS labor_cost FROM {FQ('facts_labor_daypart')} WHERE store_id = '{S1}' AND date = DATE'{anchor}') l")
print(f"S001 sales yesterday: {yday}   labor % of sales on anchor: {labor_pct}")
print("All checks passed.")